# Beginner Wiki surface guardrails

This notebook checks the small public teaching surface for AlphaClaw's beginner Wiki path.

It exists to answer **WHY DO WE NEED THAT?** for the tutorial itself: keep the first-use pages short, plain, and free of advanced development vocabulary.

It does not teach contributors Jupyter, Git, Python, or AlphaClaw internals.


In [ ]:
import re
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "contributor").exists():
    ROOT = ROOT.parent

PAGES = {
    "Start Here": ROOT / "contributor/wiki-templates/Start-Here.md",
    "Make Another Wiki Page": ROOT / "contributor/wiki-templates/Make-Another-Wiki-Page.md",
}

texts = {name: path.read_text(encoding="utf-8") for name, path in PAGES.items()}
texts


In [ ]:
def plain_text(markdown: str) -> str:
    text = re.sub(r"<!--.*?-->", " ", markdown, flags=re.DOTALL)
    text = re.sub(r"\[\[([^]|]+)(?:\|[^]]+)?\]\]", r"\1", text)
    text = re.sub(r"[`*_#>\[\]()|]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def syllables(word: str) -> int:
    word = re.sub(r"[^a-z]", "", word.lower())
    if not word:
        return 0
    count = len(re.findall(r"[aeiouy]+", word))
    if word.endswith("e") and not word.endswith(("le", "ye")) and count > 1:
        count -= 1
    return max(1, count)

def flesch_kincaid_grade(markdown: str) -> float:
    text = plain_text(markdown)
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    words = re.findall(r"\b[A-Za-z][A-Za-z'-]*\b", text)
    total_syllables = sum(syllables(word) for word in words)
    return 0.39 * (len(words) / len(sentences)) + 11.8 * (total_syllables / len(words)) - 15.59

def word_count(markdown: str) -> int:
    return len(re.findall(r"\b[A-Za-z][A-Za-z'-]*\b", plain_text(markdown)))


In [ ]:
ADVANCED_TERMS = {
    "branch", "pull request", "sha-256", "sha256", "ci", "fork",
    "upstream", "downstream", "provenance", "runtime", "kernel",
    "ontology", "recursive", "agent authority",
}

FORK_PHRASES = {"learn more", "curious about", "next steps"}

required_start = {
    "open source", "MIT License", "This Wiki is public", "Save Page",
    "Gemini", "Make Another Wiki Page",
}

checks = []
for name, text in texts.items():
    lower = text.casefold()
    grade = flesch_kincaid_grade(text)
    words = word_count(text)
    found_advanced = sorted(term for term in ADVANCED_TERMS if term in lower)
    found_forks = sorted(term for term in FORK_PHRASES if term in lower)
    checks.append((name, "grade <= 8", grade <= 8, round(grade, 1)))
    checks.append((name, "no advanced terms", not found_advanced, found_advanced))
    checks.append((name, "no optional learning forks", not found_forks, found_forks))
    limit = 450 if name == "Start Here" else 180
    checks.append((name, f"word count <= {limit}", words <= limit, words))

for phrase in required_start:
    checks.append(("Start Here", f"contains: {phrase}", phrase in texts["Start Here"], phrase))

failed = [row for row in checks if not row[2]]
for row in checks:
    print("PASS" if row[2] else "FAIL", "|", row[0], "|", row[1], "|", row[3])

assert not failed, f"Beginner surface guardrails failed: {failed}"
